# Part 6 — The forward process: destroying data on schedule

_Rigorous Courses · Diffusion Models — Part 6 of 12_

**A fixed noising recipe, and a one-line shortcut that jumps any image to any noise level**

You will build the noise schedule (`betas`, `alphas`, `abar`), watch a real digit dissolve into static, and verify numerically that noising step by step and jumping straight to step $t$ give the exact same distribution — the closed form the whole field runs on. Every claim from the lesson gets a numeric check here.

---

This notebook accompanies the lesson. Run cells top to bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)

## The schedule: betas, alphas, abar

The forward process is fully specified by its list of noise doses $\beta_1, \dots, \beta_T$. From it we compute the survival factors $\alpha_t = 1 - \beta_t$ and the cumulative survival $\bar\alpha_t = \prod_{s=1}^{t} \alpha_s$ — the fraction of the original signal's variance that survives to step $t$.

We use $T = 200$ here (the size we will train with in part 10) and the standard linear ramp of doses from $10^{-4}$ to $0.02$. One bookkeeping note: Python arrays are 0-indexed, so `abar[i]` holds $\bar\alpha_{i+1}$. We also build `abar_full`, where `abar_full[t]` is exactly $\bar\alpha_t$, including the convention $\bar\alpha_0 = 1$ (before any steps, all of the signal survives).

### Step 1 — Build the schedule arrays

Three lines of numpy. Look at the printout: $\bar\alpha_t$ starts a hair below 1 and only ever falls — every dose destroys a little more, and destruction never reverses. The asserts pin down both facts.

In [ ]:
T = 200
betas = np.linspace(1e-4, 0.02, T)
alphas = 1.0 - betas
abar = np.cumprod(alphas)

# abar_full[t] = abar_t, with the convention abar_0 = 1 (no steps taken yet).
abar_full = np.concatenate(([1.0], abar))

print(f"beta_1 = {betas[0]:.6f}    beta_T = {betas[-1]:.6f}")
print(f"abar_1 = {abar[0]:.6f}    abar_T = {abar[-1]:.6f}")

assert np.all(np.diff(abar) < 0), "abar must strictly decrease"
assert np.all((abar > 0) & (abar < 1))

## Watching a digit dissolve

Time to see the forward process do its work on a real image. We take one handwritten digit (8×8 pixels) and noise it to six different levels with the lesson's one-line sampler,

$$ x_t = \sqrt{\bar\alpha_t}\, x_0 + \sqrt{1-\bar\alpha_t}\,\epsilon, \qquad \epsilon \sim \mathcal{N}(0, \mathbf{I}). $$

No loops over steps — each panel is a single draw at its own $t$.

### Step 2 — Load one digit and scale it to [-1, 1]

The sklearn digits have pixel values 0 to 16. We rescale so mid-gray sits at 0 — the "mean about 0, spread about 1" convention the variance bookkeeping in the lesson assumed. The asserts confirm the shape and the range.

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
img = digits.images[3]
x0 = img / 8.0 - 1.0

print(f"image shape: {x0.shape}")
print(f"pixel range after scaling: [{x0.min():.2f}, {x0.max():.2f}]")

assert x0.shape == (8, 8)
assert x0.min() >= -1.0 and x0.max() <= 1.0

### Step 3 — The iconic strip: one image, six noise levels

We reuse a single noise image $\epsilon$ across all six panels, so you watch the very same noise pattern grow while the very same digit fades — a clean visual of the blend $\sqrt{\bar\alpha_t}$ signal + $\sqrt{1-\bar\alpha_t}$ noise. Below the plot we print the noise share $1-\bar\alpha_t$ per panel and assert it grows along the strip.

In [ ]:
t_list = [0, 25, 50, 100, 150, 199]
eps_img = rng.standard_normal(x0.shape)

fig, axes = plt.subplots(1, 6, figsize=(12, 2.6))
for ax, t in zip(axes, t_list):
    ab = abar_full[t]
    xt = np.sqrt(ab) * x0 + np.sqrt(1.0 - ab) * eps_img
    ax.imshow(xt, cmap="gray", vmin=-2.5, vmax=2.5)
    ax.set_title(f"t = {t}\nabar = {ab:.3f}", fontsize=9)
    ax.axis("off")
fig.suptitle("one digit, noised to six levels by the one-line sampler")
plt.tight_layout()
plt.show()

noise_share = 1.0 - abar_full[t_list]

print("noise share 1 - abar_t per panel:", np.round(noise_share, 3))

assert np.all(np.diff(noise_share) > 0)

## Two roads to step t: the key check

The lesson proved that these two procedures produce the same distribution:

- **Road A (stepwise):** apply the kernel $x_t = \sqrt{1-\beta_t}\, x_{t-1} + \sqrt{\beta_t}\,\epsilon_t$ one hundred times, with fresh noise every step.
- **Road B (one shot):** apply the closed form $x_t = \sqrt{\bar\alpha_t}\, x_0 + \sqrt{1-\bar\alpha_t}\,\epsilon$ once.

The proof leaned on part 3's sum rule at exactly one spot — merging independent Gaussian doses. If that argument is right, 5,000 samples down each road must have matching means, matching variances, and overlapping histograms. Let's find out.

### Step 4 — Noise 5,000 copies of the same point, step by step

Every sample starts at the same clean value $x_0 = 2$ and gets its own independent 100-step noise journey. The predicted end distribution is $\mathcal{N}(\sqrt{\bar\alpha_{100}} \cdot 2,\ 1-\bar\alpha_{100})$.

In [ ]:
n = 5000
t_target = 100
x0_val = 2.0

xt = np.full(n, x0_val)
for i in range(t_target):
    eps_step = rng.standard_normal(n)
    xt = np.sqrt(alphas[i]) * xt + np.sqrt(betas[i]) * eps_step
xt_stepwise = xt

print(f"stepwise  mean = {xt_stepwise.mean():.4f}   var = {xt_stepwise.var():.4f}")

### Step 5 — One shot with the closed form, then compare

One draw per sample, no loop. Compare the two empirical means and variances with each other and with the theory values $\sqrt{\bar\alpha_t}\, x_0$ and $1 - \bar\alpha_t$. Four asserts, all with a 0.05 tolerance for sampling wiggle.

In [ ]:
ab_t = abar_full[t_target]
eps = rng.standard_normal(n)
xt_oneshot = np.sqrt(ab_t) * x0_val + np.sqrt(1.0 - ab_t) * eps

mean_pred = np.sqrt(ab_t) * x0_val
var_pred = 1.0 - ab_t

print(f"one-shot  mean = {xt_oneshot.mean():.4f}   var = {xt_oneshot.var():.4f}")
print(f"predicted mean = {mean_pred:.4f}   var = {var_pred:.4f}")

assert abs(xt_stepwise.mean() - xt_oneshot.mean()) < 0.05
assert abs(xt_stepwise.var() - xt_oneshot.var()) < 0.05
assert abs(xt_stepwise.mean() - mean_pred) < 0.05
assert abs(xt_stepwise.var() - var_pred) < 0.05

### Step 6 — Overlay the histograms

Numbers matching is good; shapes matching is better. Both histograms should sit on top of each other and on top of the predicted Gaussian curve. The closed form is not an approximation of the chain — it is the chain, compressed into one draw.

In [ ]:
grid = np.linspace(xt_stepwise.min(), xt_stepwise.max(), 200)
pdf_pred = np.exp(-(grid - mean_pred) ** 2 / (2 * var_pred)) / np.sqrt(2 * np.pi * var_pred)

plt.figure(figsize=(7, 4))
plt.hist(xt_stepwise, bins=60, density=True, alpha=0.5, label="stepwise (100 doses)")
plt.hist(xt_oneshot, bins=60, density=True, alpha=0.5, label="one-shot (closed form)")
plt.plot(grid, pdf_pred, "k--", label="predicted Gaussian")
plt.xlabel("x_t value")
plt.ylabel("density")
plt.title(f"t = {t_target}: two roads, one distribution")
plt.legend()
plt.show()

## The T = 3 hand example, verified

The lesson's running example — shared with parts 7, 8, and 9 — uses $T = 3$ with $\beta = (0.1,\ 0.2,\ 0.3)$, giving $\alpha = (0.9,\ 0.8,\ 0.7)$ and $\bar\alpha = (0.9,\ 0.72,\ 0.504)$. With $x_0 = 2$, both roads gave $q(x_2 \mid x_0) = \mathcal{N}(1.6971,\ 0.28)$. Let's make the computer redo every one of those hand calculations.

### Step 7 — Recompute the lesson's numbers

Road A tracks the mean and variance dose by dose (mean gets scaled by $\sqrt{\alpha_t}$; variance gets scaled by $\alpha_t$, then the fresh $\beta_t$ is added). Road B jumps with the closed form. The asserts demand agreement with each other and with the lesson's 4-decimal values.

In [ ]:
betas3 = np.array([0.1, 0.2, 0.3])
alphas3 = 1.0 - betas3
abar3 = np.cumprod(alphas3)

print("alphas:", alphas3)
print("abar  :", abar3)

assert np.allclose(abar3, [0.9, 0.72, 0.504])

# Road A — track mean and variance step by step from the fixed point x0 = 2.
m = 2.0
v = 0.0
for i in range(2):
    m = np.sqrt(alphas3[i]) * m
    v = alphas3[i] * v + betas3[i]

# Road B — the closed form in one jump.
m_closed = np.sqrt(abar3[1]) * 2.0
v_closed = 1.0 - abar3[1]

print(f"stepwise    mean = {m:.4f}   var = {v:.4f}")
print(f"closed form mean = {m_closed:.4f}   var = {v_closed:.4f}")

assert np.isclose(m, m_closed)
assert np.isclose(v, v_closed)
assert abs(m_closed - 1.6971) < 5e-4
assert abs(v_closed - 0.28) < 1e-12

## Schedules: linear vs cosine (T = 1000)

Now the real thing: $T = 1000$. The lesson's two-line hand estimate said $\log\bar\alpha_T \approx -\sum_t \beta_t = -10.05$, so $\bar\alpha_T \approx e^{-10.05} \approx 4.3 \times 10^{-5}$, and promised the exact product is about $4.04 \times 10^{-5}$. We also build the cosine schedule, which chooses the $\bar\alpha_t$ curve directly as a smooth cosine arc instead of ramping the doses.

### Step 8 — Build both schedules

Watch the printout: the exact linear $\bar\alpha_T$ and the hand estimate agree to within about 7% — the price of the $\log(1-x) \approx -x$ shortcut. Either way, only about 0.00004 of the image survives.

In [ ]:
T_real = 1000
betas_lin = np.linspace(1e-4, 0.02, T_real)
abar_lin = np.cumprod(1.0 - betas_lin)

# Cosine schedule (Nichol & Dhariwal 2021): pick the abar curve directly.
s = 0.008
t_grid = np.arange(T_real + 1) / T_real
f = np.cos((t_grid + s) / (1.0 + s) * np.pi / 2.0) ** 2
abar_cos = f[1:] / f[0]

hand_estimate = np.exp(-np.sum(betas_lin))

print(f"linear abar_T exact    = {abar_lin[-1]:.2e}")
print(f"linear abar_T estimate = {hand_estimate:.2e}   (log(1-x) ~ -x shortcut)")
print(f"cosine abar_T exact    = {abar_cos[-1]:.2e}")

assert abar_lin[-1] < 1e-4, "the linear schedule should destroy almost everything"
assert np.all(np.diff(abar_cos) < 0)

### Step 9 — Plot the survival curves and the SNR

Left: $\bar\alpha_t$ for both schedules. The linear curve idles near 1 for its opening stretch and then dives; the cosine curve spreads the destruction more evenly. Right: $\mathrm{SNR}(t) = \bar\alpha_t/(1-\bar\alpha_t)$ on a log scale — a fall of many orders of magnitude. We also print when each schedule's image is "basically gone" ($\bar\alpha_t < 0.01$).

In [ ]:
steps_axis = np.arange(1, T_real + 1)
snr_lin = abar_lin / (1.0 - abar_lin)
snr_cos = abar_cos / (1.0 - abar_cos)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(steps_axis, abar_lin, label="linear")
ax1.plot(steps_axis, abar_cos, label="cosine")
ax1.set_xlabel("step t")
ax1.set_ylabel("abar_t")
ax1.set_title("surviving signal fraction")
ax1.legend()

ax2.semilogy(steps_axis, snr_lin, label="linear")
ax2.semilogy(steps_axis, snr_cos, label="cosine")
ax2.set_ylim(1e-6, 1e5)
ax2.set_xlabel("step t")
ax2.set_ylabel("SNR(t), log scale")
ax2.set_title("signal-to-noise ratio")
ax2.legend()

plt.tight_layout()
plt.show()

gone_lin = int(steps_axis[abar_lin < 0.01][0])
gone_cos = int(steps_axis[abar_cos < 0.01][0])

print(f"abar_t drops below 0.01 at t = {gone_lin} (linear) vs t = {gone_cos} (cosine)")

## The end state: x_T is standard noise

The whole design promise: after the full schedule, the data is standard Gaussian noise, whatever it started as. With $\bar\alpha_T \approx 4 \times 10^{-5}$, a clean value of $x_0 = 2$ contributes only $\sqrt{\bar\alpha_T} \cdot 2 \approx 0.013$ to the mean — a whisper — and the variance is $1 - \bar\alpha_T \approx 1$. Part 8 will lean on exactly this when it fixes the generative model's starting distribution to $\mathcal{N}(0, \mathbf{I})$.

### Step 10 — Sample x_T and check its statistics

5,000 draws of $x_T$ from $x_0 = 2$ under the linear $T = 1000$ schedule. The asserts demand mean within 0.05 of 0 and variance within 0.05 of 1.

In [ ]:
ab_T = abar_lin[-1]
eps_T = rng.standard_normal(n)
xT = np.sqrt(ab_T) * x0_val + np.sqrt(1.0 - ab_T) * eps_T

print(f"x_T mean = {xT.mean():.4f}   (target 0)")
print(f"x_T var  = {xT.var():.4f}   (target 1)")
print(f"surviving signal fraction abar_T = {ab_T:.2e}")

assert abs(xT.mean()) < 0.05
assert abs(xT.var() - 1.0) < 0.05

## Practice

Try each one in the empty cell below it, then reveal the worked solution. These are the same problems as the lesson — redo them here with code as your calculator and checker.

**Problem 1.** A new schedule has $T = 2$ with $\beta = (0.2,\ 0.5)$. Compute $\alpha_1, \alpha_2$, then $\bar\alpha_1, \bar\alpha_2$, and finally the mean and variance of $q(x_2 \mid x_0)$ for $x_0 = 3$.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Survival factors: $\alpha_1 = 1 - 0.2 = 0.8$ and $\alpha_2 = 1 - 0.5 = 0.5$ (definition $\alpha_t = 1-\beta_t$).
- Running products: $\bar\alpha_1 = 0.8$, $\bar\alpha_2 = 0.8 \times 0.5 = 0.4$ (fractions of fractions multiply).
- Closed-form mean: $\sqrt{0.4} \times 3 = 0.6325 \times 3 = 1.8974$.
- Closed-form variance: $1 - 0.4 = 0.6$.

```python
betas_p1 = np.array([0.2, 0.5])
abar_p1 = np.cumprod(1.0 - betas_p1)
mean_p1 = np.sqrt(abar_p1[-1]) * 3.0
var_p1 = 1.0 - abar_p1[-1]

print("abar:", abar_p1)
print(f"mean = {mean_p1:.4f}   var = {var_p1:.4f}")
```

**Answer:** $\alpha = (0.8,\ 0.5)$, $\bar\alpha = (0.8,\ 0.4)$, and $q(x_2 \mid x_0 = 3) = \mathcal{N}(1.8974,\ 0.6)$.

</details>

**Problem 2.** Rerun the two-step composition with fresh numbers: $\alpha_1 = 0.96$, $\alpha_2 = 0.75$. Show by the stepwise route that the total noise variance after two steps equals $1 - \alpha_1\alpha_2$, and give the signal coefficient.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Substitute step 1 into step 2: $x_2 = \sqrt{0.75}\big(\sqrt{0.96}\, x_0 + \sqrt{0.04}\,\epsilon_1\big) + \sqrt{0.25}\,\epsilon_2$.
- Distribute: signal coefficient $\sqrt{0.75}\sqrt{0.96} = \sqrt{0.72} \approx 0.8485$.
- First noise term's variance: $0.75 \times 0.04 = 0.03$ (scaling rule: the constant enters squared).
- Second noise term's variance: $0.25$.
- Independence lets the variances add: $0.03 + 0.25 = 0.28$.
- Closed form check: $1 - \alpha_1\alpha_2 = 1 - 0.72 = 0.28$. Match.

```python
a1 = 0.96
a2 = 0.75
noise_var = a2 * (1 - a1) + (1 - a2)

print(f"stepwise noise variance = {noise_var:.4f}")
print(f"1 - a1*a2               = {1 - a1 * a2:.4f}")
```

**Answer:** total noise variance $= 0.28 = 1 - \alpha_1\alpha_2$; signal coefficient $\sqrt{0.72} \approx 0.8485$, i.e. $\bar\alpha_2 = 0.72$.

</details>

**Problem 3.** In the closed-form derivation, where exactly is the independence of $\epsilon_1$ and $\epsilon_2$ used — and why is it safe to assume?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- The move: independence is used once, when the two scaled noise terms merge into one Gaussian with variance $\alpha_2(1-\alpha_1) + (1-\alpha_2)$. Every other step is pure algebra.
- Why needed, twice over: (a) part 2's rule $\mathrm{Var}(X+Y) = \mathrm{Var}(X) + \mathrm{Var}(Y)$ requires the cross term $\mathbb{E}[XY] - \mathbb{E}[X]\mathbb{E}[Y]$ to vanish, which independence guarantees; (b) part 3's sum rule — the merged term being *Gaussian* at all — is stated for independent Gaussian summands.
- Why safe: the forward chain draws $\epsilon_t$ fresh at every step, with no look at earlier draws or earlier data. Independence is built in by construction.

**Answer:** at the "add the variances" merge; guaranteed because each dose is a fresh, independent draw by the definition of the chain.

</details>

**Problem 4.** For the running example ($\bar\alpha = (0.9,\ 0.72,\ 0.504)$), compute $\mathrm{SNR}(1)$, $\mathrm{SNR}(2)$, $\mathrm{SNR}(3)$, and interpret the value at $t = 3$.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- $\mathrm{SNR}(1) = 0.9 / 0.1 = 9$.
- $\mathrm{SNR}(2) = 0.72 / 0.28 = 2.5714$.
- $\mathrm{SNR}(3) = 0.504 / 0.496 = 1.0161$.
- Interpretation: $\mathrm{SNR}(3) \approx 1$ means signal and noise variance are nearly equal — half signal, half static.

```python
abar_p4 = np.array([0.9, 0.72, 0.504])
snr_p4 = abar_p4 / (1.0 - abar_p4)

print("SNR:", np.round(snr_p4, 4))
```

**Answer:** $\mathrm{SNR} = (9,\ 2.5714,\ 1.0161)$; by $t = 3$ signal and noise have an equal say.

</details>

**Problem 5.** Double every dose in the running example: $\beta = (0.2,\ 0.4,\ 0.6)$. Compute the new $\bar\alpha_t$, compare with the originals, and explain the small-dose rule of thumb for what doubling all $\beta$ does to $\bar\alpha_t$.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- New survivals: $\alpha = (0.8,\ 0.6,\ 0.4)$.
- New products: $\bar\alpha_1 = 0.8$; $\bar\alpha_2 = 0.8 \times 0.6 = 0.48$; $\bar\alpha_3 = 0.48 \times 0.4 = 0.192$.
- Versus the originals $(0.9,\ 0.72,\ 0.504)$: survival at $t=3$ fell from about a half to about a fifth.
- Rule of thumb: $\log\bar\alpha_t \approx -\sum_s \beta_s$, so doubling every dose doubles the log — which squares $\bar\alpha_t$. Prediction: $0.504^2 = 0.254$; truth: $0.192$. Right direction, imperfect because these toy doses are far from small. At real doses (0.0001-0.02) the squaring rule is excellent.

```python
abar_orig = np.cumprod(1.0 - np.array([0.1, 0.2, 0.3]))
abar_doubled = np.cumprod(1.0 - np.array([0.2, 0.4, 0.6]))

print("original:", abar_orig)
print("doubled :", abar_doubled)
print(f"squaring rule predicts {abar_orig[-1] ** 2:.3f}, exact is {abar_doubled[-1]:.3f}")
```

**Answer:** new $\bar\alpha = (0.8,\ 0.48,\ 0.192)$ — much faster destruction; doubling every $\beta$ approximately squares $\bar\alpha_t$, exactly so in the small-dose limit.

</details>

**Problem 6.** Show why the shrink is necessary: run the no-shrink recipe $x_t = x_{t-1} + \sqrt{\beta_t}\,\epsilon_t$ on the running example, starting from data with $\mathrm{Var}(x_0) = 1$, and contrast with the real kernel.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- No shrink, step 1: $\mathrm{Var}(x_1) = 1 + 0.1 = 1.1$ (independent noise: variances add, nothing removed first).
- Step 2: $1.1 + 0.2 = 1.3$. Step 3: $1.3 + 0.3 = 1.6$. In general the variance is $1 + \sum_s \beta_s$ — it grows without bound.
- Real kernel: each step gives $(1-\beta_t)\cdot 1 + \beta_t = 1$. The shrink removes exactly the variance the dose adds; the total never moves.
- At real scale: $\sum \beta_t \approx 10.05$, so the no-shrink end state has variance $\approx 11$ — and it depends on the schedule. Generation (part 8) needs a fixed, known start: $\mathcal{N}(0, 1)$ per coordinate, which only the shrink delivers.

```python
var_ns = 1.0
for b in [0.1, 0.2, 0.3]:
    var_ns = var_ns + b

var_real = 1.0
for b in [0.1, 0.2, 0.3]:
    var_real = (1.0 - b) * var_real + b

print(f"no-shrink variance after 3 steps: {var_ns:.1f}")
print(f"real-kernel variance after 3 steps: {var_real:.1f}")
```

**Answer:** without the shrink, variance grows to $1 + \sum_s \beta_s$ (1.6 here, about 11 on the real schedule); with the $\sqrt{1-\beta_t}$ shrink it stays exactly 1, so the end state is the standard Gaussian we need.

</details>

**Problem 7.** One-line sampler plug-in: with the running example, $x_0 = 1.5$, $t = 2$, and the noise draw $\epsilon = -0.5$, compute $x_2$ to four decimals.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Look up $\bar\alpha_2 = 0.72$.
- Signal part: $\sqrt{0.72} \times 1.5 = 0.8485 \times 1.5 = 1.2728$.
- Noise weight: $\sqrt{1 - 0.72} = \sqrt{0.28} = 0.5292$; noise part: $0.5292 \times (-0.5) = -0.2646$.
- Blend: $x_2 = 1.2728 - 0.2646 = 1.0082$.

```python
x2 = np.sqrt(0.72) * 1.5 + np.sqrt(0.28) * (-0.5)

print(f"x_2 = {x2:.4f}")
```

**Answer:** $x_2 \approx 1.0082$.

</details>

## Wrap-up

Verified in this notebook: the schedule arrays behave ($\bar\alpha_t$ strictly falls, stays in $(0,1)$); a real digit dissolves into static under the one-line sampler; noising 5,000 points through 100 individual doses and jumping with the closed form $x_t = \sqrt{\bar\alpha_t}\, x_0 + \sqrt{1-\bar\alpha_t}\,\epsilon$ produce the same mean, variance, and histogram; the $T = 3$ hand example gives $\mathcal{N}(1.6971,\ 0.28)$ by both roads; the real linear schedule ends with $\bar\alpha_T \approx 4 \times 10^{-5}$; and $x_T$ has mean 0 and variance 1, as the design promised.

Next, Part 7 runs the movie backwards: Bayes' rule plus complete-the-square, aimed at this forward kernel, yields the exact one-step denoiser $q(x_{t-1} \mid x_t, x_0)$ — perfect denoising, if only you knew $x_0$.